# Math for Data Science

This notebook covers core math concepts used in ML & AI:

- Linear algebra (vectors, matrices, transformations)
- Probability & statistics (distributions, expectations, sampling)
- Calculus fundamentals used in optimization (derivatives, gradients, gradient descent)
- Small, runnable examples showing how these ideas map to ML algorithms

Run each cell sequentially. The notebook is concise but practical — perfect as a refresher or a short workshop.


## 1 — Linear Algebra: vectors & matrices

We'll create vectors/matrices, compute dot products, matrix multiplication, and demonstrate eigenvalues / SVD.


In [ ]:
import numpy as np
# Vectors
v = np.array([1.0, 2.0, 3.0])
w = np.array([0.5, -1.0, 2.0])
print("v:", v)
print("w:", w)
print("Dot product v·w:", v.dot(w))
print("Elementwise product:", v * w)

# Matrices
A = np.array([[2.0, 1.0, 0.0],
              [0.0, 3.0, 1.0],
              [1.0, 0.0, 2.0]])
B = np.array([[1.0, 2.0, 0.0],
              [0.0, 1.0, 1.0],
              [2.0, 0.0, 1.0]])
print("\nA:\n", A)
print("\nB:\n", B)
print("\nMatrix product A @ B:\n", A @ B)

# SVD and eigenvalues
U, s, VT = np.linalg.svd(A)
eigvals, eigvecs = np.linalg.eig(A)
print("\nSVD singular values:", s)
print("\nEigenvalues:", eigvals)


### Linear transformation (visual intuition)
Apply a 2×2 matrix to a set of 2D vectors and plot the result to see stretching/rotation.


In [ ]:
import matplotlib.pyplot as plt
# 2D example
M = np.array([[1.5, 0.5],
              [0.2, 1.2]])
# grid of points (vectors)
grid = np.array([[x,y] for x in np.linspace(-2,2,9) for y in np.linspace(-2,2,9)])
transformed = grid @ M.T

plt.figure(figsize=(6,6))
plt.axhline(0, color='k', linewidth=0.5)
plt.axvline(0, color='k', linewidth=0.5)
plt.scatter(grid[:,0], grid[:,1], label='original', alpha=0.6)
plt.scatter(transformed[:,0], transformed[:,1], label='transformed', alpha=0.6)
for i in range(0, len(grid), 8):
    plt.arrow(0,0, grid[i,0], grid[i,1], head_width=0.05, alpha=0.3)
    plt.arrow(0,0, transformed[i,0], transformed[i,1], head_width=0.05, alpha=0.6, color='orange')
plt.legend()
plt.title('Linear transformation of a 2D grid by matrix M')
plt.gca().set_aspect('equal', adjustable='box')
plt.show()


## 2 — Probability & Statistics

Key concepts: expectation, variance, sampling from distributions, and the Central Limit Theorem (CLT).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
# Sample from distributions
rng = np.random.default_rng(42)
samples = rng.normal(loc=5.0, scale=2.0, size=10000)
print("Sample mean:", samples.mean())
print("Sample variance:", samples.var(ddof=0))

# Empirical probability for an event
p_empirical = np.mean(samples > 6.0)
print("P(X > 6) approx:", p_empirical)

# Central Limit Theorem demo: means of small samples
means = [rng.normal(loc=0, scale=1, size=30).mean() for _ in range(2000)]
plt.figure(figsize=(6,4))
plt.hist(means, bins=30)
plt.title('Distribution of sample means (CLT demo)')
plt.xlabel('sample mean'); plt.ylabel('frequency')
plt.show()


## 3 — Calculus: derivatives & optimization

We show derivative of a simple function and implement gradient descent to fit a linear regression model.


In [ ]:
# symbolic derivative (if sympy available) or numeric derivative
try:
    import sympy as sp
    x = sp.symbols('x')
    f = x**3 - 3*x**2 + 2*x
    print("Symbolic derivative of f(x)=x^3-3x^2+2x is:", sp.diff(f, x))
except Exception as e:
    print("sympy not available — skipping symbolic derivative:", e)

# Numeric derivative (finite differences)
def numeric_derivative(f, x, h=1e-6):
    return (f(x+h) - f(x-h)) / (2*h)

def f_num(x):
    return x**3 - 3*x**2 + 2*x

print("Numeric derivative at x=1.0:", numeric_derivative(f_num, 1.0))


### Linear regression: normal equation (closed-form) vs gradient descent

We generate synthetic data and compare solutions.


In [ ]:
# Synthetic data
rng = np.random.default_rng(1)
X = rng.normal(size=(200,2))
true_w = np.array([1.5, -2.0])
y = X @ true_w + 0.5 * rng.normal(size=200)

# Add bias column
X_b = np.hstack([np.ones((X.shape[0],1)), X])

# Closed-form (normal equation)
w_closed = np.linalg.pinv(X_b.T @ X_b) @ X_b.T @ y
print("Closed-form weights:", w_closed)

# Gradient descent (mean squared error)
def mse_grad(w, Xb, y):
    preds = Xb @ w
    error = preds - y
    grad = (2/len(y)) * (Xb.T @ error)
    return grad

# initialize
w = np.zeros(X_b.shape[1])
lr = 0.1
for i in range(2000):
    g = mse_grad(w, X_b, y)
    w = w - lr * g
print("GD weights:", w)

# Compare predictions
try:
    from sklearn.metrics import mean_squared_error
    pred_closed = X_b @ w_closed
    pred_gd = X_b @ w
    print("MSE closed-form:", mean_squared_error(y, pred_closed))
    print("MSE GD:", mean_squared_error(y, pred_gd))
except Exception as e:
    print("sklearn not available; showing squared error manually.")
    pred_closed = X_b @ w_closed
    pred_gd = X_b @ w
    print("MSE closed-form:", ((y - pred_closed)**2).mean())
    print("MSE GD:", ((y - pred_gd)**2).mean())


## 4 — Logistic regression: sigmoid, loss, and gradient

We'll implement the sigmoid, binary cross-entropy loss, and one step of gradient descent.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
# synthetic separable data
rng = np.random.default_rng(2)
N = 200
X = rng.normal(loc=0.0, scale=1.0, size=(N,2))
w_true = np.array([2.0, -1.0])
logits = X @ w_true
y = (logits + 0.5*rng.normal(size=N)) > 0
y = y.astype(float)

# add bias
Xb = np.hstack([np.ones((N,1)), X])
w = np.zeros(Xb.shape[1])

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def bce_loss(w, Xb, y):
    z = Xb @ w
    p = sigmoid(z)
    eps = 1e-9
    return -np.mean(y * np.log(p+eps) + (1-y)*np.log(1-p+eps))

def bce_grad(w, Xb, y):
    z = Xb @ w
    p = sigmoid(z)
    grad = (1/len(y)) * (Xb.T @ (p - y))
    return grad

print("Initial loss:", bce_loss(w, Xb, y))
grad = bce_grad(w, Xb, y)
w -= 0.5 * grad  # one GD update with lr=0.5
print("Loss after one update:", bce_loss(w, Xb, y))

# quick visualization of decision boundary
xx = np.linspace(X[:,0].min()-1, X[:,0].max()+1, 200)
yy = np.linspace(X[:,1].min()-1, X[:,1].max()+1, 200)
XX, YY = np.meshgrid(xx, yy)
grid = np.c_[XX.ravel(), YY.ravel()]
grid_b = np.hstack([np.ones((grid.shape[0],1)), grid])
probs = sigmoid(grid_b @ w).reshape(XX.shape)

plt.figure(figsize=(6,5))
plt.contourf(XX, YY, probs, levels=20, alpha=0.6)
plt.scatter(X[:,0], X[:,1], c=y, edgecolor='k')
plt.title('Logistic model probability surface (after one update)')
plt.show()


## Summary

- Linear algebra provides the language for data and models (vectors/matrices).
- Probability & statistics let us reason about uncertainty and sampling.
- Calculus (derivatives, gradients) gives us tools to optimize model parameters.
- Together these form the mathematical backbone of common ML algorithms.

---

**Try:** modify noise, step sizes, and dataset size to see how convergence and fit change.
